In [3]:
# Prepare for pyspark use
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

In [8]:
# Get Yellow 2025-11 data from official website
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 17:26:03--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.33.102.33, 13.33.102.12, 13.33.102.215, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.33.102.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: 'yellow_tripdata_2025-11.parquet'

     0K .......... .......... .......... .......... ..........  0% 6.91M 10s
    50K .......... .......... .......... .......... ..........  0% 9.17M 9s
   100K .......... .......... .......... .......... ..........  0%  158M 6s
   150K .......... .......... .......... .......... ..........  0% 26.7M 5s
   200K .......... .......... .......... .......... ..........  0% 11.2M 5s
   250K .......... .......... .......... .......... ..........  0% 47.3M 5s
   300K .......... .......... .......... .......... ..........  0% 43.

In [9]:
# Create dataframe from Yellow 2025-11 data
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [10]:
# Display Yellow 20205-11 data to explore dataset
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [11]:
# Reparition Yellow 2025-11 data
df = df.repartition(4)

In [12]:
# Write repartitioned data to parquet files
df.write.parquet('yellow_4/2025/11')

In [13]:
# Print schema for reference
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [19]:
# Prepare for use of sql functions
from pyspark.sql import functions as F

In [30]:
# Create new fields as date-only, converted from datetime fields
# to convert timestamp -> date using F.to_date()
df_updated = df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.tpep_pickup_datetime))

In [31]:
# Show 1st 20 rows to verify additional columns with date-only formatting
df_updated.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_date|dropoff_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|  

In [32]:
# Prepare temp table from updated schema & data
df_updated.registerTempTable('yellow_nov_data')

In [33]:
# Verify sample SQL statement with new temp table
spark.sql("""
SELECT * FROM yellow_nov_data LIMIT 10;
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_date|dropoff_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|  

In [37]:
# Q3: How many taxi trips were started on November 15th ?
spark.sql("""
SELECT  COUNT(*)
FROM yellow_nov_data 
WHERE pickup_date = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [42]:
# Q4: What is the length of the longest trip (in hours) in the dataset?
spark.sql("""
SELECT 
    MAX(timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime)) / 3600 AS longest_trip_hour
FROM yellow_nov_data;
""").show()

+-----------------+
|longest_trip_hour|
+-----------------+
|90.64666666666666|
+-----------------+



In [43]:
# Retrieve zone lookup data from official website
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-10 14:58:52--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.33.102.215, 13.33.102.12, 13.33.102.33, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.33.102.215|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: 'taxi_zone_lookup.csv'

     0K .......... ..                                         100%  353M=0s

2026-03-10 14:58:53 (353 MB/s) - 'taxi_zone_lookup.csv' saved [12331/12331]



In [47]:
# # Create dataframe from zone data, while keeping the column names and inferring the schema
df_zones = spark.read.option("header", "true").option("inferSchema", "true").csv("taxi_zone_lookup.csv")

In [48]:
# Show 1st 20 rows to explore dataset
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [53]:
# Prepare temp view from zone dataframe
df_zones.createOrReplaceTempView("zones")

In [54]:
# Q6: What is the name of the pickup location zone with the fewest records? If multiple match, select any
spark.sql("""
SELECT 
    y.PULocationID,
    z.Zone,
    COUNT(*) AS frequency
FROM yellow_nov_data AS y
JOIN zones AS z ON z.LocationID = y.PULocationID
GROUP BY y.PULocationID,z.Zone
ORDER BY frequency ASC
;
""").show()

+------------+--------------------+---------+
|PULocationID|                Zone|frequency|
+------------+--------------------+---------+
|          84|Eltingville/Annad...|        1|
|           5|       Arden Heights|        1|
|         105|Governor's Island...|        1|
|         187|       Port Richmond|        3|
|         199|       Rikers Island|        4|
|         111| Green-Wood Cemetery|        4|
|         204|   Rossville/Woodrow|        4|
|         109|         Great Kills|        4|
|           2|         Jamaica Bay|        5|
|         251|         Westerleigh|       12|
|         176|             Oakwood|       14|
|         245|       West Brighton|       14|
|         172|New Dorp/Midland ...|       14|
|          59|        Crotona Park|       14|
|         253|       Willets Point|       15|
|          27|Breezy Point/Fort...|       16|
|         206|Saint George/New ...|       17|
|          30|       Broad Channel|       18|
|         156|     Mariners Harbor